# A* Search on a Grid — With Graphical Display

_Generated: 2025-10-22T17:05:08.122183Z_

This notebook implements **A\*** search **from first principles** on a 2D grid with obstacles. It supports **4- or 8-connected** motion, **Manhattan** or **Euclidean** heuristics, and produces a **step-by-step visualization** of the search frontier, the explored set, and the final shortest path. We use only NumPy and Matplotlib.

**Features**
- Deterministic grid generator (or random map)
- A* with `heapq`, tie-breaking, and parent pointers
- Live rendering of open/closed sets and the final path
- Optional animation export (MP4) and downloadable artifacts

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports, Reproducibility, and Plot Utilities

In [ ]:
import numpy as np
import heapq
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.set_printoptions(precision=4, suppress=True)
SEED = 42
rng = np.random.default_rng(SEED)

def show_grid(grid, start, goal, title="Grid"):
    fig = plt.figure(figsize=(5,5))
    plt.imshow(grid, interpolation='nearest')
    si, sj = start; gi, gj = goal
    plt.scatter([sj],[si], marker='o')  # start
    plt.scatter([gj],[gi], marker='x')  # goal
    plt.title(title); plt.axis('off'); plt.tight_layout()
    plt.show()

## 2) Grid Generation

You can use a **predefined map** or generate a **random obstacle field**. Cells with value `1` are **walls**; `0` are **free**.

In [ ]:
@dataclass
class GridWorld:
    grid: np.ndarray  # 0 free, 1 wall
    start: tuple
    goal: tuple
    eight_connected: bool = False

def make_predefined():
    g = np.array([
        [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
        [0,1,1,1,1,0,0,0,0,1,1,1,1,1,0],
        [0,0,0,0,1,0,1,0,0,0,0,0,0,1,0],
        [0,1,0,0,1,0,1,0,1,1,1,1,0,1,0],
        [0,1,0,0,0,0,1,0,0,0,0,1,0,1,0],
        [0,1,1,1,1,0,1,1,1,1,0,1,0,1,0],
        [0,0,0,0,1,0,0,0,0,1,0,1,0,0,0],
        [0,1,1,0,1,1,1,1,0,1,0,1,1,1,0],
        [0,0,0,0,0,0,0,1,0,0,0,0,0,1,0],
        [0,1,1,1,1,1,0,1,1,1,1,1,0,1,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
    ], dtype=int)
    start = (0,0)
    goal  = (10,14)
    return GridWorld(g, start, goal, eight_connected=True)

def make_random(h=30, w=45, wall_prob=0.28):
    g = (rng.random((h,w)) < wall_prob).astype(int)
    start = (0,0); goal = (h-1, w-1)
    g[start] = 0; g[goal] = 0
    # ensure some corridor via simple carving
    for i in range(h):
        g[i, min(i, w-1)] = 0
    return GridWorld(g, start, goal, eight_connected=True)

WORLD = make_predefined()  # or: WORLD = make_random()
grid, start, goal = WORLD.grid, WORLD.start, WORLD.goal
show_grid(grid, start, goal, title="Initial Grid (o=start, x=goal)")

## 3) A* Implementation (From First Principles)

We implement A* with a binary heap (priority queue). The cost is `g + h`, where `g` is the path cost and `h` is an admissible heuristic.

In [ ]:
def neighbors(i, j, grid, eight=False):
    H, W = grid.shape
    steps4 = [(-1,0),(1,0),(0,-1),(0,1)]
    steps8 = steps4 + [(-1,-1),(-1,1),(1,-1),(1,1)]
    steps = steps8 if eight else steps4
    for di, dj in steps:
        ni, nj = i+di, j+dj
        if 0 <= ni < H and 0 <= nj < W and grid[ni,nj]==0:
            yield ni, nj, (2 if (di!=0 and dj!=0) else 1)  # diagonal step cost = 2 (scaled), orthogonal = 1

def manhattan(a, b):
    (i1, j1), (i2, j2) = a, b
    return abs(i1-i2) + abs(j1-j2)

def euclidean2(a, b):
    (i1, j1), (i2, j2) = a, b
    di = i1 - i2; dj = j1 - j2
    return di*di + dj*dj  # use squared distance to avoid sqrt (monotone with true distance)

def astar(grid, start, goal, eight=False, heuristic="manhattan"):
    H, W = grid.shape
    hfun = manhattan if heuristic=="manhattan" else euclidean2
    gscore = np.full((H,W), np.inf)
    fscore = np.full((H,W), np.inf)
    parent = np.full((H,W,2), -1, dtype=int)

    si, sj = start; gi, gj = goal
    gscore[si,sj] = 0.0
    fscore[si,sj] = hfun(start, goal)

    # Min-heap of (f, g, tie, (i,j))
    counter = 0
    heap = []
    heapq.heappush(heap, (fscore[si,sj], gscore[si,sj], counter, (si,sj)))

    open_set = set([(si,sj)])
    closed_set = set()

    # For visualization
    snapshots = []  # store (open_set copy, closed_set copy, current node (i,j))
    expanded_order = []

    while heap:
        f, g, _, (i, j) = heapq.heappop(heap)
        if (i,j) in closed_set:
            continue
        open_set.discard((i,j))
        closed_set.add((i,j))
        expanded_order.append((i,j))

        snapshots.append((set(open_set), set(closed_set), (i,j)))

        if (i,j) == (gi,gj):
            # reconstruct path
            path = []
            cur = (gi,gj)
            while cur != (-1,-1):
                path.append(cur)
                pi, pj = parent[cur]
                if pi < 0: break
                cur = (pi,pj)
            path.reverse()
            return {"path": path, "gscore": gscore, "fscore": fscore,
                    "parent": parent, "snapshots": snapshots, "expanded": expanded_order}

        for ni, nj, step_cost in neighbors(i, j, grid, eight):
            if (ni,nj) in closed_set:
                continue
            tentative_g = gscore[i,j] + step_cost
            if tentative_g < gscore[ni,nj]:
                parent[ni,nj] = [i,j]
                gscore[ni,nj] = tentative_g
                fscore[ni,nj] = tentative_g + hfun((ni,nj), (gi,gj))
                if (ni,nj) not in open_set:
                    counter += 1
                    heapq.heappush(heap, (fscore[ni,nj], gscore[ni,nj], counter, (ni,nj)))
                    open_set.add((ni,nj))

    return {"path": [], "gscore": gscore, "fscore": fscore,
            "parent": parent, "snapshots": snapshots, "expanded": expanded_order}

## 4) Run A*

In [ ]:
RES = astar(grid, start, goal, eight=WORLD.eight_connected, heuristic="manhattan")
path = RES["path"]
print("Path length (number of nodes):", len(path))
print("Reached goal:", bool(path and path[-1]==goal))

## 5) Visualization — Step-by-Step and Final Path

We render the grid as an image. Then we overlay: **closed set** (expanded), **open set** (frontier), **current node**, and the **final path**.

In [ ]:
def draw_state(grid, start, goal, open_set, closed_set, current=None, path=None, title="A* Step"):
    H, W = grid.shape
    img = np.zeros((H,W))
    img[:,:] = 0.9  # base
    img[grid==1] = 0.2  # walls

    # closed and open overlays (avoid setting specific colors; relative intensity only)
    for (i,j) in closed_set:
        if grid[i,j]==0:
            img[i,j] = 0.6
    for (i,j) in open_set:
        if grid[i,j]==0 and (i,j) not in closed_set:
            img[i,j] = 0.75

    if path:
        for (i,j) in path:
            img[i,j] = 0.3

    fig = plt.figure(figsize=(5,5))
    plt.imshow(img, interpolation='nearest')
    si, sj = start; gi, gj = goal
    plt.scatter([sj],[si], marker='o')   # start
    plt.scatter([gj],[gi], marker='x')   # goal
    if current is not None:
        ci, cj = current
        plt.scatter([cj],[ci], marker='s')  # current
    plt.title(title); plt.axis('off'); plt.tight_layout()
    plt.show()

# Preview a few steps
snaps = RES["snapshots"]
if snaps:
    for t in [0, len(snaps)//3, 2*len(snaps)//3, len(snaps)-1]:
        open_t, closed_t, cur = snaps[t]
        draw_state(grid, start, goal, open_t, closed_t, current=cur, title=f"A* Step t={t}")

# Final path plot
draw_state(grid, start, goal, set(), set(), current=None, path=path, title="Final Path")

## 6) Optional Animation (Matplotlib)

We build an animation across A* snapshots and save an MP4 (requires ffmpeg in Colab environments).

In [ ]:
from matplotlib import animation

def animate_search(grid, start, goal, snapshots, path=None, interval=120):
    fig = plt.figure(figsize=(5,5))
    ax = plt.gca()
    H, W = grid.shape
    base = np.zeros((H,W)); base[:,:] = 0.9; base[grid==1] = 0.2
    im = plt.imshow(base, interpolation='nearest', animated=True)
    si, sj = start; gi, gj = goal
    scat_start = plt.scatter([sj],[si], marker='o')
    scat_goal  = plt.scatter([gj],[gi], marker='x')
    scat_cur   = plt.scatter([], [], marker='s')
    plt.axis('off')

    def frame(k):
        open_k, closed_k, cur = snapshots[k]
        img = base.copy()
        for (i,j) in closed_k:
            if grid[i,j]==0: img[i,j] = 0.6
        for (i,j) in open_k:
            if grid[i,j]==0 and (i,j) not in closed_k: img[i,j] = 0.75
        if path is not None and k == len(snapshots)-1:
            for (i,j) in path: img[i,j] = 0.3
        im.set_array(img)
        if cur is not None:
            ci, cj = cur
            scat_cur.set_offsets([[cj, ci]])
        return [im, scat_start, scat_goal, scat_cur]

    anim = animation.FuncAnimation(fig, frame, frames=len(snapshots), interval=interval, blit=True)
    plt.close(fig)
    return anim

if RES["snapshots"]:
    anim = animate_search(grid, start, goal, RES["snapshots"], path=path, interval=100)
    try:
        from IPython.display import HTML
        HTML(anim.to_jshtml())
    except Exception as e:
        print("Inline animation preview not available:", e)

## 7) Save Artifacts & Download

We save the grid, start/goal, the path, and an MP4 animation (if possible). Use the helper to download a ZIP.

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/astar_run.npz",
         grid=grid, start=np.array(start), goal=np.array(goal),
         path=np.array(path, dtype=int))

# Try to save an MP4
try:
    anim = animate_search(grid, start, goal, RES["snapshots"], path=path, interval=80)
    anim.save("artifacts/astar_animation.mp4")
    print("Saved animation to artifacts/astar_animation.mp4")
except Exception as e:
    print("Could not save MP4 animation (ffmpeg likely missing or environment-limited):", e)

# Save static PNGs
def save_frame(img, fname, title):
    fig = plt.figure(figsize=(5,5))
    plt.imshow(img, interpolation='nearest'); plt.title(title); plt.axis('off'); plt.tight_layout()
    fig.savefig(f"artifacts/{fname}", dpi=120); plt.close(fig)

# Save initial, mid, final snapshots
if RES["snapshots"]:
    t0 = 0; tm = len(RES["snapshots"])//2; tL = len(RES["snapshots"])-1
    for t, label in [(t0, "t0"), (tm, "tmid"), (tL, "tlast")]:
        open_t, closed_t, cur = RES["snapshots"][t]
        H, W = grid.shape
        base = np.zeros((H,W)); base[:,:] = 0.9; base[grid==1] = 0.2
        for (i,j) in closed_t:
            if grid[i,j]==0: base[i,j] = 0.6
        for (i,j) in open_t:
            if grid[i,j]==0 and (i,j) not in closed_t: base[i,j] = 0.75
        save_frame(base, f"astar_{label}.png", f"A* {label}")
else:
    print("No snapshots to save as PNGs.")

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 8) Exercises & Extensions

- Add **weighted A*** (inflate the heuristic) and compare solution quality vs speed.
- Implement **D* Lite** for dynamic obstacles.
- Add **cost maps** (terrain costs) and visualize g-score heatmaps.
- Switch to **bidirectional A*** or **fringe search**.
- Export **GIF** instead of MP4 for environments without ffmpeg.